## Comparison of UTA and AHP Rankings

Both methods were applied to the same 26 European countries and 8 criteria.
Below we compare the resulting rankings using Kendall's τ and inspect where
the methods agree or diverge most.

In [1]:
import sys
import pathlib
import numpy as np
import pandas as pd
from scipy.stats import kendalltau
import pulp

PROJECT_ROOT = pathlib.Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from common.config import DATASET_FILE, METADATA_FILE
from common.data_loading import load_data, load_preferences, load_removal_indices
from common.uta_core import compute_characteristic_points, solve_model
from uta_discrimination.solver import get_consistent_preferences, build_discrimination_model, rank_alternatives
from ahp.hierarchy_setup import CRITERIA, CATEGORIES, CATEGORY_CRITERIA, DIRECTIONS
from ahp.weights import ahp_weights
from ahp.dm_matrices import MATRICES
from ahp.global_weights import compute_global_weights
from ahp.scoring import compute_ahp_scores

import pandas as pd
df, directions = load_data()

In [2]:
preferences     = load_preferences()
removal_indices = load_removal_indices()
criteria        = list(directions.keys())
consistent_prefs = get_consistent_preferences(preferences, removal_indices)
char_points      = compute_characteristic_points(df, directions)
model_uta, u_uta, epsilon_uta = build_discrimination_model(df, directions, consistent_prefs)
solve_model(model_uta)
uta_ranking = rank_alternatives(df, directions, u_uta, criteria, char_points)
uta_ranking.columns = ['Country', 'U(a)']
uta_ranking

,Country,U(a)
Rank,,
1,Switzerland,0.881850
2,Iceland,0.758484
3,Netherlands,0.758397
4,Norway,0.705292
5,Denmark,0.682792
6,Germany,0.681938
7,Sweden,0.660292
8,Luxembourg,0.625872
9,Finland,0.574974


In [3]:
results = {name: ahp_weights(A) for name, A in MATRICES.items()}
w_goal = results["Goal"]["weights"]
w_cats = {cat: results[cat]["weights"] for cat in CATEGORIES}
global_weights = compute_global_weights(w_goal, w_cats)

scores = compute_ahp_scores(df, global_weights)
ahp_ranking = pd.DataFrame({
    "Country":   df["Country"].values,
    "AHP Score": scores,
}).sort_values("AHP Score", ascending=False).reset_index(drop=True)
ahp_ranking.index += 1
ahp_ranking.index.name = "AHP Rank"
ahp_ranking

,Country,AHP Score
AHP Rank,,
1,Iceland,0.090508
2,Switzerland,0.088341
3,Luxembourg,0.077722
4,Netherlands,0.069335
5,Denmark,0.067769
6,Norway,0.055017
7,Germany,0.054431
8,Finland,0.046423
9,Austria,0.045818


In [12]:
uta = uta_ranking.reset_index().rename(columns={"Rank": "UTA Rank"})
ahp = ahp_ranking.reset_index().rename(columns={"AHP Rank": "AHP Rank"})
merged = uta.merge(ahp[["Country", "AHP Rank"]], on="Country")

tau, p_val = kendalltau(merged["UTA Rank"], merged["AHP Rank"])
print(f"Kendall's τ = {tau:.4f}  (p = {p_val:.4f})\n")
print(merged[["UTA Rank", "AHP Rank", "Country"]].sort_values("UTA Rank").to_string(index=False))

Kendall's τ = 0.8031  (p = 0.0000)

 UTA Rank  AHP Rank         Country
        1         2     Switzerland
        2         1         Iceland
        3         4     Netherlands
        4         6          Norway
        5         5         Denmark
        6         7         Germany
        7        10          Sweden
        8         3      Luxembourg
        9         8         Finland
       10         9         Austria
       11        16         Estonia
       12        15  United Kingdom
       13        12         Ireland
       14        14  Czech Republic
       15        11         Belgium
       16        19       Lithuania
       17        18        Slovenia
       18        21          Latvia
       19        13          Poland
       20        17          France
       21        22           Spain
       22        20         Hungary
       23        25        Portugal
       24        23           Italy
       25        24 Slovak Republic
       26        26         

Kendall's τ = **0.80** (p < 0.001) — strong agreement between the two methods.

In [11]:
merged["Shift"] = merged["AHP Rank"] - merged["UTA Rank"]

uta_top3 = merged.nsmallest(3, "UTA Rank")["Country"].values
ahp_top3 = merged.nsmallest(3, "AHP Rank")["Country"].values
uta_bot3 = merged.nlargest(3, "UTA Rank")["Country"].values
ahp_bot3 = merged.nlargest(3, "AHP Rank")["Country"].values

print(f"Top 3 UTA:    {', '.join(uta_top3)}")
print(f"Top 3 AHP:    {', '.join(ahp_top3)}")
print(f"Bottom 3 UTA: {', '.join(uta_bot3)}")
print(f"Bottom 3 AHP: {', '.join(ahp_bot3)}")

print("\nBiggest disagreements (|shift| ≥ 5):")
big = merged[merged["Shift"].abs() >= 5].sort_values("Shift")
print(big[["Country", "UTA Rank", "AHP Rank", "Shift"]].to_string(index=False))

Top 3 UTA:    Switzerland, Iceland, Netherlands
Top 3 AHP:    Iceland, Switzerland, Luxembourg
Bottom 3 UTA: Greece, Slovak Republic, Italy
Bottom 3 AHP: Greece, Portugal, Slovak Republic

Biggest disagreements (|shift| ≥ 5):
   Country  UTA Rank  AHP Rank  Shift
    Poland        19        13     -6
Luxembourg         8         3     -5
   Estonia        11        16      5


### Conclusions

Both methods place the same cluster of Northern/Western European countries at the top and Southern/Eastern Europe at the bottom. Greece ranks last in both; Switzerland and Iceland occupy the top two spots (just swapped).

**Main disagreements:**
- **Luxembourg** drops from 3rd (AHP) to 8th (UTA) — AHP rewards it heavily for outstanding earnings (~47% global weight), while UTA distributes importance more evenly.
- **Estonia** rises from 16th (AHP) to 11th (UTA) for the opposite reason: good scores on pollution and working hours matter more in UTA.

**Remarks:** In AHP, personal earnings alone carries nearly half the total weight, which is why high-earning countries dominate. UTA reflects a more balanced view — no single criterion overwhelms the rest.